In [ ]:
!pip install kaggle -q

In [ ]:
import os
import zipfile
from google.colab import drive
import shutil

drive.mount('/content/drive')
zip_files = [f"/content/drive/MyDrive/images_product_phase{i}.zip" for i in range(1, 16)]
target_folder = "images_product"
#temp_extract = "temp_extract"
os.makedirs(target_folder, exist_ok=True)

Mounted at /content/drive


In [ ]:
for zip_path in zip_files:
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            for member in zip_ref.infolist():
                if not member.is_dir():
                    filename = os.path.basename(member.filename)
                    if filename:
                        target_path = os.path.join(target_folder, filename)
                        with open(target_path, "wb") as f:
                            f.write(zip_ref.read(member))
            print(f"Finished processing & merging: {os.path.basename(zip_path)}")
    else:
        print(f"Warning: File {zip_path} not found!")

Finished processing & merging: images_product_phase1.zip
Finished processing & merging: images_product_phase2.zip
Finished processing & merging: images_product_phase3.zip
Finished processing & merging: images_product_phase4.zip
Finished processing & merging: images_product_phase5.zip
Finished processing & merging: images_product_phase6.zip
Finished processing & merging: images_product_phase7.zip
Finished processing & merging: images_product_phase8.zip
Finished processing & merging: images_product_phase9.zip
Finished processing & merging: images_product_phase10.zip
Finished processing & merging: images_product_phase11.zip
Finished processing & merging: images_product_phase12.zip
Finished processing & merging: images_product_phase13.zip
Finished processing & merging: images_product_phase14.zip
Finished processing & merging: images_product_phase15.zip


In [ ]:
total_images = len([f for f in os.listdir(target_folder) if os.path.isfile(os.path.join(target_folder, f))])
print(f"Total images in folder: {total_images}")

Total images in folder: 3467990


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/tokopedia_products.csv")
df.shape

(3614813, 6)

In [ ]:
existing_images = {f.replace('.jpg', '') for f in os.listdir('images_product') if f.endswith('.jpg')}
missing_id_product = df[~df["ID_Product"].astype(str).isin(existing_images)].copy()
missing_id_product.head()

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product
42,43,Apotek Sengeti Farma Sekernan,ALLOPURINOL 100MG 1 STRIP 10 TABLET (Gen HJ),Rp4.976,NaN,https://www.tokopedia.com/apoteksengetifarmase...
43,44,Apotek Sengeti Farma Sekernan,INTERHISTIN 50MG 1 STRIP ISI 10 TABLET,Rp16.813,NaN,https://www.tokopedia.com/apoteksengetifarmase...
44,45,Apotek Sengeti Farma Sekernan,IMODIUM 2MG 1 STRIP 10 TABLET,Rp143.783,NaN,https://www.tokopedia.com/apoteksengetifarmase...
45,46,Apotek Sengeti Farma Sekernan,FARMOTEN 25MG 1 STRIP 10 TABLET,Rp5.499,NaN,https://www.tokopedia.com/apoteksengetifarmase...
51,52,Apotek Sengeti Farma Sekernan,"HARNAL OCAS 0,4MG 1 BLISTER 10 TABLET",Rp161.068,NaN,https://www.tokopedia.com/apoteksengetifarmase...


In [ ]:
missing_id_product.shape

(146762, 6)

In [ ]:
import requests
import pandas as pd
import os, re, time, threading, math
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from google.colab import files
import json

In [ ]:
df_sid = pd.read_csv("/content/drive/MyDrive/Scrapping_list_shop_with_SID.csv")
df1 = missing_id_product.merge(df_sid[['Nama Toko', 'SID']], left_on='Shop_Name', right_on='Nama Toko', how='left')
df1.drop(columns=['Nama Toko'], inplace=True)

In [ ]:
df1.shape

(146762, 7)

In [ ]:
path_folder = '/content/images_product'
batch_size  = 500
thread_count1 = 20
os.makedirs(path_folder, exist_ok=True)
write_lock = threading.Lock()
error_list = []

headers = {
    'sec-ch-ua-platform': '"Windows"',
    'x-version': '17e0d90',
    'Referer': 'https://www.tokopedia.com/',
    'sec-ch-ua': '"Google Chrome";v="147", "Not.A/Brand";v="8", "Chromium";v="147"',
    'x-price-center': 'true',
    'sec-ch-ua-mobile': '?0',
    'bd-device-id': '7628337271406577160',
    'x-source': 'tokopedia-lite',
    'x-device': 'default_v3',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
    'accept': '*/*',
    'content-type': 'application/json',
    'x-tkpd-lite-service': 'zeus'
}

In [ ]:
def mark_error(product_id):
    with write_lock:
        error_list1.append(product_id)

def clean_url(url):
    return url.split('?')[0].rstrip('/')

def get_keyword(product_name):
    words = str(product_name).split()
    return ' '.join(words[:3])

def get_image_url(sid, product_name, target_url, retries=3, session=None):
    if session is None:
        session = requests.Session()
    keyword      = get_keyword(product_name)
    target_clean = clean_url(target_url)
    payload = [{
        "operationName": "ShopProducts",
        "variables": {
            "source": "shop", "sid": str(sid), "page": 1, "perPage": 80,
            "keyword": keyword, "etalaseId": "etalase", "sort": 1,
            "user_districtId": "2274", "user_cityId": "176",
            "user_lat": "0", "user_long": "0",
            "usecase": "ace_get_shop_product_v2"
        },
        "query": "query ShopProducts($sid: String!, $source: String, $page: Int, $perPage: Int, $keyword: String, $etalaseId: String, $sort: Int, $user_districtId: String, $user_cityId: String, $user_lat: String, $user_long: String, $usecase: String) {\n  GetShopProduct(shopID: $sid, source: $source, filter: {page: $page, perPage: $perPage, fkeyword: $keyword, fmenu: $etalaseId, sort: $sort, user_districtId: $user_districtId, user_cityId: $user_cityId, user_lat: $user_lat, user_long: $user_long, usecase: $usecase}) {\n    data {\n      product_url\n      primary_image { original }\n    }\n  }\n}\n"
    }]
    for _ in range(retries):
        try:
            res = session.post(
                'https://gql.tokopedia.com/graphql/ShopProducts',
                headers=headers, json=payload, timeout=15
            )
            if res.status_code != 200:
                continue
            products = res.json()[0]['data']['GetShopProduct']['data']
            if not products:
                return None
            for p in products:
                if clean_url(p.get('product_url', '')) == target_clean:
                    return p.get('primary_image', {}).get('original')
            return products[0].get('primary_image', {}).get('original')
        except Exception:
            pass
    return None

def process_row(row):
    session      = requests.Session()
    product_id   = str(row["ID_Product"])
    sid          = str(row["SID"])
    product_name = row["Product_Name"]
    url          = row["URL_Product"]
    filepath     = os.path.join(path_folder, f"{product_id}.jpg")
    if os.path.exists(filepath):
        return "skip"
    img_url = get_image_url(sid, product_name, url, session=session)
    if not img_url:
        mark_error(product_id)
        return "error"
    for _ in range(3):
        try:
            img_res = session.get(
                img_url,
                headers={'User-Agent': headers['User-Agent']},
                timeout=10
            )
            if img_res.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(img_res.content)
                return "success"
        except Exception:
            pass
    mark_error(product_id)
    return "error"

In [ ]:
df_todo = df1.copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count1) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 146762/146762 [39:03<00:00, 62.63product/s, batch=294/294, Success: =6969, Fail: =139793]


In [ ]:
sorted(error_list, key=int)[:5]

['43', '44', '45', '46', '52']

In [ ]:
df1[df1["ID_Product"].astype(str).isin(error_list)].copy()

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product,SID
0,43,Apotek Sengeti Farma Sekernan,ALLOPURINOL 100MG 1 STRIP 10 TABLET (Gen HJ),Rp4.976,NaN,https://www.tokopedia.com/apoteksengetifarmase...,17115861
1,44,Apotek Sengeti Farma Sekernan,INTERHISTIN 50MG 1 STRIP ISI 10 TABLET,Rp16.813,NaN,https://www.tokopedia.com/apoteksengetifarmase...,17115861
2,45,Apotek Sengeti Farma Sekernan,IMODIUM 2MG 1 STRIP 10 TABLET,Rp143.783,NaN,https://www.tokopedia.com/apoteksengetifarmase...,17115861
3,46,Apotek Sengeti Farma Sekernan,FARMOTEN 25MG 1 STRIP 10 TABLET,Rp5.499,NaN,https://www.tokopedia.com/apoteksengetifarmase...,17115861
4,52,Apotek Sengeti Farma Sekernan,"HARNAL OCAS 0,4MG 1 BLISTER 10 TABLET",Rp161.068,NaN,https://www.tokopedia.com/apoteksengetifarmase...,17115861
...,...,...,...,...,...,...,...
146757,3614448,Apotek Ubay Farma by GoApotik,ERPHAFLAM 50 MG BOX 50 TABLET,Rp23.747,5.0,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
146758,3614472,Apotek Ubay Farma by GoApotik,CESTER PER BOX ISI 30 KAPSUL,Rp101.070,NaN,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
146759,3614498,Apotek Ubay Farma by GoApotik,RANTIN 150 MG STRIP 10 TABLET,Rp73.063,5.0,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250
146760,3614633,Apotek Ubay Farma by GoApotik,SURBEX Z STRIP ISI 6 TABLET,Rp29.225,5.0,https://www.tokopedia.com/apotek-ubay-farma-by...,15047250


In [ ]:
error_list1 = []
df_todo = df1[df1["ID_Product"].astype(str).isin(error_list)].copy()
rows    = df_todo.to_dict("records")
total   = len(rows)
n_batch = math.ceil(total / batch_size)
success_count = error_count = skip_count = 0

with tqdm(total=total, unit="product") as pbar:
    for batch_idx in range(n_batch):
        batch = rows[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        with ThreadPoolExecutor(max_workers=thread_count1) as executor:
            futures = {executor.submit(process_row, row): row for row in batch}
            for future in as_completed(futures):
                result = future.result()
                if result == "success":
                    success_count += 1
                elif result == "error":
                    error_count += 1
                else:
                    skip_count += 1
                pbar.update(1)
                pbar.set_postfix({
                    "batch": f"{batch_idx+1}/{n_batch}",
                    "Success: ": success_count,
                    "Fail: ": error_count
                })
        del batch, futures

100%|██████████| 139793/139793 [34:16<00:00, 67.99product/s, batch=280/280, Success: =129, Fail: =139664]


In [ ]:
total_images1 = len([f for f in os.listdir(target_folder) if os.path.isfile(os.path.join(target_folder, f))])
print(f"Total images in folder: {total_images1}")

Total images in folder: 3475088


In [ ]:
import zipfile
import os
from tqdm import tqdm

folder   = '/content/images_product'
zip_path = '/content/tokopedia_images_product.zip'

images = [f for f in os.listdir(folder) if f.endswith('.jpg')]
total  = len(images)

with zipfile.ZipFile(zip_path, 'a', compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in tqdm(images, unit="gambar"):
        filepath = os.path.join(folder, filename)
        try:
            zf.write(filepath, arcname=filename)
            os.remove(filepath)
        except Exception:
            pass

print(f"Selesai: {total:,} gambar")
print(f"Ukuran zip: {os.path.getsize(zip_path)/1024/1024/1024:.2f} GB")

100%|██████████| 3475088/3475088 [3:00:48<00:00, 320.32gambar/s] 


Selesai: 3,475,088 gambar
Ukuran zip: 39.38 GB


In [ ]:
with open('/content/kaggle.json', 'r') as f:
    kaggle_api = json.load(f)

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_api, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
dataset_metadata = {
    "title": "Tokopedia Images Product",
    "id": f"{kaggle_api['username']}/tokopedia-images-product",
    "licenses": [{"name": "CC0-1.0"}]
}

with open('/content/dataset-metadata.json', 'w') as f:
    json.dump(dataset_metadata, f)

In [ ]:
!kaggle datasets create -p /content --file tokopedia_images_product.zip

usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,benchmarks,b,config,auth}
              ...
kaggle: error: unrecognized arguments: --file tokopedia_images_product.zip


In [ ]:
import os, shutil

os.makedirs('/content/kaggle_upload', exist_ok=True)
shutil.copy('/content/tokopedia_images_product.zip', '/content/kaggle_upload/')
shutil.copy('/content/dataset-metadata.json', '/content/kaggle_upload/')

!kaggle datasets create -p /content/kaggle_upload

Starting upload for file tokopedia_images_product.zip
100% 39.4G/39.4G [31:30<00:00, 22.4MB/s]
Upload successful: tokopedia_images_product.zip (39GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/fati22/tokopedia-images-product
